In [7]:
import os
import pandas as pd
from datasets import load_dataset
import torch

# Set CUDA device to cuda:5 if available
if torch.cuda.is_available():
    torch.cuda.set_device(5)
    print(f"Using CUDA device: {torch.cuda.get_device_name(5)}")
else:
    print("CUDA is not available. Running on CPU.")


# Define dataset URLs and load them
dataset_urls = [
    "ShivomH/Mental-Health-Conversations",
    "YvvonM/mental_health_data",
    "usham/mental-health-companion-new",
    "Harshallama/mental_health_alpaca_format"
]

# List to store individual DataFrames
dfs = []

# Download and process each dataset
for url in dataset_urls:
    # Load dataset from Hugging Face
    dataset = load_dataset(url)
    
    # Convert to pandas DataFrame
    df = pd.DataFrame(dataset['train'])
    
    # Rename columns if necessary
    if 'response' in df.columns:
        df = df.rename(columns={'response': 'output'})
    
    # Ensure the DataFrame has the required columns
    required_columns = ['instruction', 'input', 'output']
    df = df[required_columns] if all(col in df.columns for col in required_columns) else df
    
    dfs.append(df)

# Concatenate all DataFrames
concatenated_df = pd.concat(dfs, ignore_index=True)

# Save the concatenated dataset to ./Gemma/therapist_dataset.csv
concatenated_df.to_csv("./therapist_dataset.csv", index=False)

print("Datasets downloaded and concatenated successfully into ./Gemma/therapist_dataset.csv")

Using CUDA device: NVIDIA RTX A6000


Repo card metadata block was not found. Setting CardData to empty.


Datasets downloaded and concatenated successfully into ./Gemma/therapist_dataset.csv


In [8]:
# Load the concatenated dataset and print the number of rows
import pandas as pd
df = pd.read_csv("./therapist_dataset.csv")
print(f"Number of rows in therapist_dataset.csv: {len(df)}")

/tmp/ipykernel_1811782/3411598960.py:3: DtypeWarning: Columns (0,1,2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("./therapist_dataset.csv")


Number of rows in therapist_dataset.csv: 4378805


In [9]:
df = df.dropna(subset=['instruction', 'input', 'output'])
df.head()

,instruction,input,output,Instruction,Input,Response,Output
0,You are an empathetic and supportive AI chatbo...,I've been feeling so sad and overwhelmed latel...,"Hey there, I'm here to listen and support you....",NaN,NaN,NaN,NaN
1,You are an empathetic and supportive AI chatbo...,"I recently got a promotion at work, which I th...",I can understand how it can be overwhelming wh...,NaN,NaN,NaN,NaN
2,You are an empathetic and supportive AI chatbo...,"Well, the workload has increased significantly...",It sounds like you're dealing with a lot of pr...,NaN,NaN,NaN,NaN
3,You are an empathetic and supportive AI chatbo...,I've been trying to prioritize my tasks and de...,It's great to hear that you're already impleme...,NaN,NaN,NaN,NaN
4,You are an empathetic and supportive AI chatbo...,You're right. I haven't really opened up about...,"It's completely normal to feel that way, but r...",NaN,NaN,NaN,NaN


In [3]:
# 1. Drop duplicate columns by lowercasing all names
df = df.loc[:, ~df.columns.duplicated()]

# 2. Drop rows that are exact duplicates of column names
df = df[~df['instruction'].str.lower().eq('instruction')]

# 3. Drop rows where all elements are NaN (like the ones at the bottom)
df = df.dropna(how='all')

# 4. Optional: Reset the index
df = df.reset_index(drop=True)

# 5. View the cleaned dataframe
print(df.head())

                                         instruction  \
0  You are an empathetic and supportive AI chatbo...   
1  You are an empathetic and supportive AI chatbo...   
2  You are an empathetic and supportive AI chatbo...   
3  You are an empathetic and supportive AI chatbo...   
4  You are an empathetic and supportive AI chatbo...   

                                               input  \
0  I've been feeling so sad and overwhelmed latel...   
1  I recently got a promotion at work, which I th...   
2  Well, the workload has increased significantly...   
3  I've been trying to prioritize my tasks and de...   
4  You're right. I haven't really opened up about...   

                                              output Instruction Input  \
0  Hey there, I'm here to listen and support you....         NaN   NaN   
1  I can understand how it can be overwhelming wh...         NaN   NaN   
2  It sounds like you're dealing with a lot of pr...         NaN   NaN   
3  It's great to hear that you

In [4]:
df.head()

,instruction,input,output,Instruction,Input,Response,Output
0,You are an empathetic and supportive AI chatbo...,I've been feeling so sad and overwhelmed latel...,"Hey there, I'm here to listen and support you....",NaN,NaN,NaN,NaN
1,You are an empathetic and supportive AI chatbo...,"I recently got a promotion at work, which I th...",I can understand how it can be overwhelming wh...,NaN,NaN,NaN,NaN
2,You are an empathetic and supportive AI chatbo...,"Well, the workload has increased significantly...",It sounds like you're dealing with a lot of pr...,NaN,NaN,NaN,NaN
3,You are an empathetic and supportive AI chatbo...,I've been trying to prioritize my tasks and de...,It's great to hear that you're already impleme...,NaN,NaN,NaN,NaN
4,You are an empathetic and supportive AI chatbo...,You're right. I haven't really opened up about...,"It's completely normal to feel that way, but r...",NaN,NaN,NaN,NaN


In [6]:
%pip install soxr 

from unsloth import FastLanguageModel
from transformers import AutoTokenizer
import torch

# Load model and tokenizer using Unsloth
model_name = "belal212/therapist-gemma"

# Load tokenizer separately
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model using Unsloth
model, _ = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,  # This can help with memory and compatibility
    device_map = {"": "cuda:5"}  # Force model to use cuda:5 like your setup
)

# Disable torch compile to avoid recompile limit issues
torch._dynamo.config.suppress_errors = True
torch._dynamo.reset()

# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

# Encode input prompt
prompt = "Hello, I'm feeling stressed and anxious. What should I do?"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda:5")

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=100, 
        do_sample=True, 
        top_p=0.95, 
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id
    )

# Decode and print response
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.
==((====))==  Unsloth 2025.7.11: Fast Gemma3N patching. Transformers: 4.54.1.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 8. Max memory: 47.402 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3N does not support SDPA - switching to eager!


Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.21it/s]


Hello, I'm feeling stressed and anxious. What should I do?

It's completely understandable to feel stressed and anxious. It's a very common human experience. Here's a breakdown of things you can do, ranging from immediate self-soothing to longer-term strategies.  I'll also try to keep it concise and actionable.

**1. Immediate Self-Soothing (5-10 minutes):**

*   **Deep Breathing:**  Focus on slow, deep breaths. Inhale deeply through your nose, hold for a
